# 02 · Analyze sales growth and gross profit
Consume only the successful, checksum-verified output of notebook 01 from this exact run.
Revenue = quantity × unit price; gross profit = revenue − quantity × unit cost.
Gross margin uses total profit / total revenue, rather than averaging order-level margins.
This is descriptive analysis, not a causal claim or forecast.

In [ ]:
# The runner injects these values. For manual use, copy run_id from notebook 01.
run_id = None
input_csv = None
demo_failure = None

In [ ]:
from notebook_pipeline import start_notebook

session = start_notebook("02_analyze", run_id=run_id)
run_id = session.run_id
import pandas as pd

from notebook_pipeline.runtime import write_json

In [ ]:
# step: load_clean_sales
sales = pd.read_parquet(session.input("01_ingest", "clean_sales.parquet"))
sales.head()

In [ ]:
# step: calculate_order_economics
if demo_failure == "unexpected":
    raise RuntimeError("Deliberate analysis failure: testing automatic cell traceback logging.")
sales = sales.assign(
    revenue=lambda df: df["quantity"] * df["unit_price"],
    cost=lambda df: df["quantity"] * df["unit_cost"],
    month=lambda df: df["order_date"].dt.to_period("M").dt.to_timestamp(),
)
sales["gross_profit"] = sales["revenue"] - sales["cost"]

## Monthly performance
Reindex the full month sequence so a missing month is explicit. Growth is undefined for
the first month and after a zero-revenue month; preserve those values as missing.

In [ ]:
# step: monthly_summary
monthly = sales.groupby("month").agg(
    revenue=("revenue", "sum"),
    gross_profit=("gross_profit", "sum"),
    orders=("order_id", "nunique"),
    units=("quantity", "sum"),
)
months = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS", name="month")
monthly = monthly.reindex(months, fill_value=0)
monthly["gross_margin"] = monthly["gross_profit"].div(monthly["revenue"].replace(0, float("nan")))
previous_revenue = monthly["revenue"].shift(1).replace(0, float("nan"))
monthly["revenue_growth"] = monthly["revenue"].div(previous_revenue).sub(1)
monthly = monthly.reset_index()
monthly.round(3)

In [ ]:
# step: category_summary
categories = sales.groupby("category", as_index=False).agg(
    revenue=("revenue", "sum"),
    gross_profit=("gross_profit", "sum"),
    orders=("order_id", "nunique"),
)
categories["gross_margin"] = categories["gross_profit"] / categories["revenue"]
categories = categories.sort_values("revenue", ascending=False).reset_index(drop=True)
categories.round(3)

In [ ]:
# step: summarize_kpis
kpis = {
    "orders": int(sales["order_id"].nunique()),
    "revenue": round(float(sales["revenue"].sum()), 2),
    "gross_profit": round(float(sales["gross_profit"].sum()), 2),
    "gross_margin": float(sales["gross_profit"].sum() / sales["revenue"].sum()),
    "top_category": str(categories.iloc[0]["category"]),
}
session.log.info("analysis.summarized", extra={"fields": {"orders": kpis["orders"]}})
kpis

In [ ]:
# step: publish_analysis
with session.output("monthly_sales.parquet") as path:
    monthly.to_parquet(path, index=False)
with session.output("category_sales.parquet") as path:
    categories.to_parquet(path, index=False)
with session.output("kpis.json") as path:
    write_json(path, kpis)

In [ ]:
# step: finish_analysis
print(f"Use run_id = {run_id!r} in notebook 03.")
session.finish()